# Ch.2  データを「見る」

受講生用ノートブック

| 項目 | 内容 |
|------|------|
| 使うデータ | ワインデータ（Ch.1 と同じ）178件・13特徴量・3品種 |
| 演習時間 | 40分（問3まで完了で合格） |
| ゴール | グラフから「この変数が品種を決めている」という仮説を立てる |

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇のグラフを描きたい / 〇〇を可視化したい
【使うライブラリ】matplotlib / seaborn
【データの形】df という DataFrame、178行×14列（数値13列 + 品種名列）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

「グラフを描いて」ではなく、
「df の alcohol 列と proline 列を散布図で描きたい。品種名ごとに色を変えたい。どう書くか」のように、
**列名・グラフの種類・色分けの条件** をセットで伝えましょう。

---
## 準備  ライブラリとデータを読み込む

⛔ 注意 最初にこのセルを実行してください。

In [ ]:
import pandas as pd                    # データ操作の基本ライブラリ
import matplotlib.pyplot as plt        # グラフ描画の基本ライブラリ
import seaborn as sns                  # 統計グラフ（ヒートマップ等）に特化したライブラリ
import japanize_matplotlib             # matplotlib で日本語を表示できるようにする
from sklearn.datasets import load_wine # ワインデータセットを読み込む関数

# データを読み込んで DataFrame を作成する
wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種名"] = ["wine_" + str(t) for t in wine.target]

---
## STEP 1  データの「分布の形」を見る

なぜ分布を確認するのか：
「平均値」は一点の代表値に過ぎません。
値がどの範囲に集まっているか・偏りがあるか・外れ値がないかは、グラフで見るとわかります。

---

Q1-1：アルコール度数（alcohol）はどんな分布をしていますか？

ヒストグラムを描いて、値の広がり方・集中具合を確認してください。

💡 ヒント：「特定の列をヒストグラムで表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# alcohol（アルコール度数）のヒストグラムを描いて、値の分布を確認する
plt.hist(df["alcohol"], bins=20)
plt.title("alcohol の分布")        # グラフのタイトル
plt.xlabel("アルコール度数")        # X 軸のラベル
plt.ylabel("件数")                  # Y 軸のラベル
plt.show()                          # グラフを表示する

---
STEP 1（追加）  Ch.1 で「std 最大」だった proline の分布を確認しましょう

Ch.1 の `describe()` で proline の std が断トツ1位でした。
同じヒストグラムを proline で描いて、alcohol と形がどう違うか比べてみましょう。

💡 ヒント：「特定の列をヒストグラムで表示する方法」を Copilot に聞いてみましょう（列名を proline に変えるだけです）。

In [ ]:
# proline のヒストグラムを描いて、alcohol との分布の形の違いを比較する
plt.hist(df["proline"], bins=20, edgecolor="black")  # edgecolor → 棒の枠線を黒にする
plt.title("proline の分布")
plt.xlabel("proline")
plt.ylabel("件数")
plt.show()

---
STEP 1（まとめ）  品種ごとに色分けしてヒストグラムを重ねましょう

alcohol の分布で「山が2〜3つに見えた」のは、複数の品種が混ざっているからです。
品種別に色を分けてヒストグラムを重ねると、その理由が一目でわかります。

📌 コンセプトスライドの「2つの山 = 2グループが混在している」を実際に確認してください。
→ STEP 2 の箱ひげ図で「品種ごとに分けて見る」ことの意味につながります。

💡 ヒント：「品種ごとにループして同じグラフに hist() を重ねる方法（alpha で透明度を指定）」を Copilot に聞いてみましょう。

In [ ]:
# 品種ごとに色を分けてヒストグラムを重ねて描く（「山の数 = 品種の混在」を確認する）
for name in df["品種名"].unique():
    group = df[df["品種名"] == name]          # 1品種のデータだけを取り出す
    plt.hist(group["alcohol"], bins=15, alpha=0.5, label=name)
    # alpha=0.5 → 透明度50%にして、重なりが見えるようにする

plt.xlabel("alcohol（アルコール度数）")
plt.ylabel("件数")
plt.title("品種別 alcohol 分布（重ね書き）")
plt.legend()   # 凡例（どの色がどの品種かを表示）
plt.show()

---
### STEP 1 の気づき（AI 禁止：グラフを見てメモ）

Q：ヒストグラムの山はいくつありましたか？その形から何が想像できますか？

>

---
## STEP 2  品種ごとに「差」を比べる

なぜ品種別で比べるのか：
Ch.1 の集計で「品種によって平均が違う」ことがわかりました。
箱ひげ図を使うと「中央値の違い」「ばらつきの大きさ」「外れ値の有無」をグループ別に一度に確認できます。

---

Q2-1：品種ごとに alcohol の分布を比べてみましょう

3品種（wine_0・wine_1・wine_2）ごとにアルコール度数の分布を比べてください。

💡 ヒント：「グループ別に特定の列の箱ひげ図を描く方法」を Copilot に聞いてみましょう。

In [ ]:
# 品種ごとの alcohol の分布を箱ひげ図で比較する
df.boxplot(column="alcohol", by="品種名")
plt.suptitle("")                         # pandas が自動で付けるタイトルを消す
plt.title("品種別 alcohol の分布")
plt.ylabel("アルコール度数")
plt.show()

---
STEP 2（追加）  proline も品種別に比べてみましょう

alcohol の箱ひげ図と見比べて「どちらが品種を分けやすいか」を視覚的に判断してください。

📌 仮説：「proline は std が大きかった → 品種間の差が大きいのでは？」

💡 ヒント：alcohol を proline に変えるだけです。Copilot に聞いてみましょう。

In [ ]:
# 品種ごとに proline の分布を箱ひげ図で比較して、alcohol との「分かれやすさ」を比べる
df.boxplot(column="proline", by="品種名")
plt.title("品種別 proline 分布")
plt.suptitle("")
plt.ylabel("proline")
plt.show()

---
### STEP 2 の気づき（AI 禁止：グラフを見てメモ）

Q：3品種のうち、最もアルコール度数に差があるペアはどれですか？また、見分けにくそうな品種ペアはどれですか？

>

---
### STEP 2（追加）の気づき（AI 禁止：1行でOK）

Q：alcohol と proline、どちらが品種を見分けやすそうでしたか？グラフから判断した理由も書いてください。

>

---
## 問1  2つの変数の「関係」をグラフで確認する ★☆☆

なぜ散布図を使うのか：
箱ひげ図は 1 変数。散布図は 2 変数の「組み合わせ」で見ます。
「alcohol が高いと proline も高い品種がある」という関係は、散布図で初めて見えます。

---

Q：alcohol（X軸）と proline（Y軸）の散布図を、品種別に色を分けて描いてみましょう

3品種が色分けされた散布図を描いて、品種がグラフ上で「分離しているか」を確認してください。

📌 なぜ alcohol と proline？：Ch.1 の groupby で wine_0 の proline が他品種の約 2 倍あり、alcohol も品種間の差が大きかった変数です。「差が大きい 2 変数の組み合わせ」を散布図で見ます。

💡 ヒント：「グループ別に色分けした散布図を描く方法」を Copilot に聞いてみましょう。

In [ ]:
# 品種別に色を分けた散布図を描いて、品種がグラフ上で分離しているか確認する
for name in df["品種名"].unique():
    group = df[df["品種名"] == name]          # 1品種のデータだけを取り出す
    plt.scatter(group["alcohol"], group["proline"], label=name, alpha=0.7)
    # alpha=0.7 → 点を少し透明にして重なりが見えるようにする

plt.xlabel("alcohol（アルコール度数）")
plt.ylabel("proline（プロリン）")
plt.title("alcohol vs proline（品種別）")
plt.legend()   # 凡例を表示
plt.show()

---
問1（追加）  品種ごとの平均を「棒グラフ」で見てみましょう

Ch.1 の `groupby` で計算した「品種ごとの平均」を、今度はグラフで表示します。
数値の表より「棒グラフ」の方が差の大きさが直感的に伝わります。

💡 ヒント：「groupby で品種ごとの proline 平均を計算して棒グラフで表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 品種ごとの proline 平均を計算する（棒グラフで使う値を準備する）
mean_proline = df.groupby("品種名")["proline"].mean()   # グループ別に平均を計算
print(mean_proline.round(1))

In [ ]:
# 品種別 proline 平均を棒グラフで表示する（Ch.1 の groupby の数値を可視化する）
mean_proline.plot(kind="bar")      # kind="bar" → 棒グラフ
plt.title("品種別 proline 平均")
plt.ylabel("proline 平均")
plt.xticks(rotation=0)             # X 軸のラベルを横向きのまま表示
plt.show()

---
### 問1 の気づき（AI 禁止：1行でOK）

Q：散布図で3品種は分かれていましたか？どのくらい分離していましたか？

>

---
## 問2  全変数の「関係の強さ」を一度に確認する ★★☆

なぜ相関ヒートマップを使うのか：
13 列もある変数を 1 つずつ散布図で確認するのは時間がかかります。
ヒートマップを使うと「強く相関する変数ペア」を一度に発見できます。

---

📌 相関係数とは：2つの変数が「一緒に増える・一緒に減る」関係の強さを **-1〜+1** の数値で表したもの。  
- **+1 に近い**：一方が増えるともう一方も増える（正の相関）  
- **-1 に近い**：一方が増えるともう一方は減る（負の相関）  
- **0 に近い**：ほぼ無関係  
目安：**絶対値が 0.7 以上**なら「強い相関あり」と判断します。

---

Q2-0：ヒートマップを描く前に「数値列」だけを確認しましょう

相関係数は数値列同士でしか計算できません。「品種名」のような文字列列を含めるとエラーになります。
まず「どの列が数値型か」を確認してから、数値列だけで相関を計算する習慣をつけましょう。

📌 Ch.3 で機械学習の特徴量 X を作るときも「品種名列を除く」という操作をします。ここで先に理解しておくとスムーズです。

💡 ヒント：「DataFrame から数値型の列だけを取り出す方法」を Copilot に聞いてみましょう。

In [ ]:
# 数値型の列だけを取り出して、ヒートマップで使える列名と列数を確認する
num_cols = df.select_dtypes(include="number").columns.tolist()

---
Q2-1：全数値列の相関係数をヒートマップで表示してみましょう

ヒートマップを描いて、**赤いマス（+0.7 以上）や青いマス（-0.7 以下）** を 1 つ見つけて、「それは何と何の変数ペアか」を確認してください。

💡 ヒント：「全数値列の相関行列をヒートマップで表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 全数値列の相関行列を計算する
corr = df.select_dtypes(include="number").corr()

In [ ]:
# 相関行列をヒートマップで可視化する（赤=正の相関、青=負の相関）
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".1f", cmap="coolwarm")
# annot=True → 各セルに数値を表示、cmap="coolwarm" → 赤青のカラーマップ
plt.title("相関ヒートマップ")
plt.tight_layout()
plt.show()

---
問2（追加）  強い相関のペアを「数値」で上位から確認しましょう

ヒートマップは全体の傾向を見るのに便利ですが、「何位が何と何のペアか」は数値で確認するほうが明確です。
相関係数の絶対値が 0.7 以上のペアを探してみましょう。

💡 ヒント：「相関行列から絶対値の大きいペアを上位で一覧する方法」を Copilot に聞いてみましょう。

In [ ]:
# 相関行列を計算する（ペア探索で使う変数として保存する）
import numpy as np
corr = df.select_dtypes(include="number").corr()

In [ ]:
# 対角線と重複を除いて、相関が強いペアを上位5件表示する

# ① 上三角行列のマスクを作る（対角線・下三角を除外して重複ペアを防ぐ）
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)

# ② マスクを使ってペア形式に変換し、絶対値で降順ソートする
corr_pairs = (
    corr.where(mask)              # 下三角と対角線を NaN にする
    .stack()                       # 行列を「ペア × 相関係数」の Series に変換
    .abs()                         # 絶対値を取る
    .sort_values(ascending=False)  # 大きい順に並べ替え
)
print("相関係数（絶対値）上位 5 ペア:")
print(corr_pairs.head(5).round(3))

---
### 問2 の気づき（AI 禁止：1行でOK）

Q：最も強い相関があった変数ペアはどれですか？（ヒートマップの赤いマスを見て確認）

>

---
## 問3  グラフから「仮説」を言葉にする ★★☆（AI 禁止・最重要）

実務でのイメージ：
グラフは「見るもの」ではなく「仮説を立てるツール」です。
「この変数が品種を決めているのでは？」という仮説を言葉にする練習をします。

---

⛔ 注意 AI は使わないこと。グラフを見て、自分の言葉で書きましょう。

💡 書き方の例：「〇〇のグラフを見ると、〇〇（品種）は〇〇（変数）が他品種より大きい／小さい。そのため〇〇が品種を分けやすい変数と考える。」

【仮説1】品種を最も分けやすそうな変数はどれですか？その理由は？

>

【仮説2】機械学習のモデル（Ch.3）で特に重要になりそうな変数を 2〜3 個予想してください。

>

【確認】Ch.3 の特徴量重要度で予想と合っていたかを確認しましょう（Ch.3 後に記入）

>

---
## 問4（発展）  pairplot で全変数を一覧する ★★★

📋 発展 時間が余った人だけ取り組んでください

---

Q：主要な変数に絞った pairplot を描いてみましょう

alcohol・proline・flavanoids・color_intensity の 4 変数を使って、
品種別色分けの pairplot（散布図行列）を描いてください。

📌 なぜこの 4 変数？：Ch.1 の集計（groupby）と Ch.2 の箱ひげ図・散布図で品種間の差が大きく見えた変数を選んでいます。自分で「他の変数でも試してみる」のも歓迎です。

💡 ヒント：「複数の変数を使った散布図行列（pairplot）を描く方法」を Copilot に聞いてみましょう。

In [ ]:
# 主要な 4 変数に絞って品種別色分けの散布図行列（pairplot）を描く
cols = ["alcohol", "proline", "flavanoids", "color_intensity", "品種名"]
sns.pairplot(df[cols], hue="品種名")
plt.show()

---
## お疲れさまでした！

| グラフ | 発見 | Ch.3 への接続 |
|-------|------|-------------|
| ヒストグラム | alcohol に複数の山 → 品種の混在 | -- |
| 箱ひげ図 | wine_0 と wine_1 は明確に分離 | 分類しやすい変数の候補 |
| 散布図 | alcohol × proline で 3品種がほぼ分離 | 機械学習の特徴量として有力 |
| 相関ヒートマップ | flavanoids と total_phenols が強相関（≈0.9） | 変数選択の参考に |

「グラフで立てた仮説」を Ch.3 の特徴量重要度で検証します！